# AiSynth Labs · AI DRUG guide · Notebook 1
## Python and data

**Author:** Toluwanimi Odunewu (@toluwanimiso)  
**Last updated:** 2026-10-30  
**Runtime:** about 6 to 10 minutes · **GPU needed:** no  
**Projects covered:** 3 to 6 (dataset, filters, clusters, analogues)  
**Licence:** CC BY 4.0

Part of the free "Get started in AI drug discovery" guide from AiSynth Labs. Run the cells from top to bottom. Change one thing at a time and write down what moved.

### Aim of this notebook
Pull a real bioactivity dataset from ChEMBL for one protein, clean it, see its chemical families and find analogues of a known drug. You will finish with four posts you can defend.

**You need:** nothing installed. The notebook installs RDKit on Colab or Kaggle by itself.

### References
1. Zdrazil B et al. (2023). *The ChEMBL Database in 2023.* Nucleic Acids Res. [doi:10.1093/nar/gkad1004](https://doi.org/10.1093/nar/gkad1004). Data: CC BY-SA 3.0.
2. Lipinski CA et al. (1997). *Experimental and computational approaches to estimate solubility and permeability in drug discovery and development settings.* Adv Drug Deliv Rev 23:3-25 (the rule of five).
3. Baell JB, Holloway GA (2010). *New substructure filters for removal of pan assay interference compounds (PAINS).* J Med Chem 53:2719-2740.
4. Brenk R et al. (2008). *Lessons learnt from assembling screening libraries for drug discovery for neglected diseases.* ChemMedChem 3:435-444 [doi:10.1002/cmdc.200700139](https://doi.org/10.1002/cmdc.200700139). The 105 unwanted-substructure alerts in RDKit's BRENK catalog.
5. Butina D (1999). *Unsupervised data base clustering based on Daylight's fingerprint and Tanimoto similarity.* J Chem Inf Comput Sci 39:747-750 [doi:10.1021/ci9803381](https://doi.org/10.1021/ci9803381).
6. RDKit: open-source cheminformatics, [rdkit.org](https://www.rdkit.org).
7. PubChem PUG REST, [pubchem.ncbi.nlm.nih.gov](https://pubchem.ncbi.nlm.nih.gov/docs/pug-rest).

---
# Practical

In [ ]:
# --- SETUP: runs on Colab, Kaggle or on your own computer -------------------------------
import os, sys, random, importlib.util
ON_COLAB = 'google.colab' in sys.modules
ON_KAGGLE = os.path.exists('/kaggle/input')
if (ON_COLAB or ON_KAGGLE) and importlib.util.find_spec('rdkit') is None:
    print('Installing RDKit (about a minute) ...')
    !pip install -q rdkit
SEED = 42                      # fixed seed: you and everyone else should get the same numbers
random.seed(SEED)
import numpy as np
np.random.seed(SEED)
print(f'Colab: {ON_COLAB} | Kaggle: {ON_KAGGLE} | seed {SEED}')

In [ ]:
# --- IMPORTS AND SETTINGS ----------------------------------------------------------------
import json, time, urllib.parse, urllib.request
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import Descriptors, Lipinski, Draw, rdFingerprintGenerator, rdFMCS
from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.ML.Cluster import Butina
RDLogger.DisableLog('rdApp.*')

# >>> Change these two lines to work on another protein. CDK5 is CHEMBL4036, BACE1 is CHEMBL4822.
TARGET_NAME = 'ACHE'
TARGET_CHEMBL_ID = 'CHEMBL220'          # human acetylcholinesterase
PIC50_RANGE = (2, 12)                   # outside this range (IC50 below 1 picomolar or above 10 millimolar) a value is treated as an error

results = {}                            # every number you will want to post is saved here

## Project 2 (optional, after the web step). The same three drugs as numbers
SwissADME gave you its calls. RDKit computes weight, logP and polar surface area from the structure alone, so you can put the two views side by side.

In [ ]:
# --- PROJECT 2 (optional): metformin, donepezil and memantine as numbers --------------------------------------------
drugs = {'metformin': 'CN(C)C(=N)N=C(N)N',
         'donepezil': 'COC1=C(C=C2C(=C1)CC(C2=O)CC3CCN(CC3)CC4=CC=CC=C4)OC',
         'memantine': 'CC12CC3CC(C1)(CC(C3)(C2)N)C'}
rows = []
for name, smi in drugs.items():
    m = Chem.MolFromSmiles(smi)
    rows.append(dict(drug=name, weight=round(Descriptors.MolWt(m), 2), logP=round(Descriptors.MolLogP(m), 2), polar_surface_area=round(Descriptors.TPSA(m), 1)))
display(pd.DataFrame(rows))
results['p2'] = rows

## Project 3. Build the dataset
ChEMBL stores one row per measurement. We ask for exact IC50 binding values in nM, turn each into a pIC50 (9 minus log10 of the IC50 in nM, so bigger means more potent), and keep one row per molecule. Every step that drops rows is counted, so you can tell the reader what you removed.

In [ ]:
# --- PROJECT 3: build the dataset ----------------------------------------------------------
CHEMBL = 'https://www.ebi.ac.uk/chembl/api/data/'

def get_json(url, tries=4):
    for k in range(tries):
        try:
            return json.load(urllib.request.urlopen(url, timeout=90))
        except Exception:
            time.sleep(2 + 3 * k)
    raise RuntimeError('could not reach ' + url)

def fetch_ic50(chembl_id):
    """Every exact IC50 binding record (in nM) for one ChEMBL target, page by page."""
    rows, offset = [], 0
    while True:
        q = urllib.parse.urlencode({'target_chembl_id': chembl_id, 'standard_type': 'IC50', 'standard_relation': '=',
                                    'assay_type': 'B', 'standard_units': 'nM', 'limit': 1000, 'offset': offset})
        page = get_json(CHEMBL + 'activity.json?' + q)
        rows += page['activities']
        if not page['page_meta']['next']:
            return pd.DataFrame(rows)
        offset += 1000

raw = fetch_ic50(TARGET_CHEMBL_ID)
data_log = {'records from ChEMBL': len(raw)}

df = raw[['molecule_chembl_id', 'canonical_smiles', 'standard_value', 'document_year']].copy()
df['standard_value'] = pd.to_numeric(df['standard_value'], errors='coerce')
df = df.dropna(subset=['standard_value', 'canonical_smiles']);  data_log['with a value and a structure'] = len(df)
df = df[df['standard_value'] > 0];                              data_log['value above zero'] = len(df)
df['pIC50'] = 9 - np.log10(df['standard_value'])                # IC50 in nM -> pIC50 (bigger = more potent)
# one row per molecule: the MEDIAN pIC50 of its records (the first record is an arbitrary choice)
df = (df.groupby('molecule_chembl_id', as_index=False)
        .agg(smiles=('canonical_smiles', 'first'), pIC50=('pIC50', 'median'), n_records=('pIC50', 'size'), document_year=('document_year', 'min')))
data_log['one row per molecule'] = len(df)
df['mol'] = [Chem.MolFromSmiles(s) for s in df['smiles']]
df = df[df['mol'].notna()].reset_index(drop=True);              data_log['structure RDKit can read'] = len(df)

# Values that are not believable: look at them, then drop them.
odd = df[~df.pIC50.between(*PIC50_RANGE)].sort_values('pIC50')
results_odd = [[r.molecule_chembl_id, round(r.pIC50, 2)] for r in odd.itertuples()]
print(f'{len(odd)} molecules have a pIC50 outside {PIC50_RANGE}. Examples:'); display(odd[['molecule_chembl_id', 'pIC50', 'n_records']].head(6)); display(odd[['molecule_chembl_id', 'pIC50', 'n_records']].tail(6))
df = df[df.pIC50.between(*PIC50_RANGE)].reset_index(drop=True); data_log['plausible pIC50 (2 to 12)'] = len(df)

print('DATA LOG, target', TARGET_NAME, TARGET_CHEMBL_ID)
for k, v in data_log.items():
    print(f'  {k:<32}{v:>8,}')
print(f"  pIC50: min {df.pIC50.min():.2f}, median {df.pIC50.median():.2f}, max {df.pIC50.max():.2f}")
results['p3'] = dict(data_log, pIC50_min=round(df.pIC50.min(), 2), pIC50_median=round(df.pIC50.median(), 2), pIC50_max=round(df.pIC50.max(), 2),
                     very_potent_over_9=int((df.pIC50 > 9).sum()))

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.hist(df.pIC50, bins=40, color='black')
ax.set_xlabel('pIC50'); ax.set_ylabel('molecules'); ax.set_title(f'{TARGET_NAME}: {len(df):,} molecules (ChEMBL, {time.strftime("%d %b %Y")})')
plt.show()

results['p3']['implausible_dropped'] = len(odd); results['p3']['implausible_examples'] = results_odd[:3] + results_odd[-3:]

## Project 4. Clean the library
Three filters, each with a reason: the **rule of five** (is it drug-like in size and polarity?), **PAINS** (does it tend to look active in many assays for the wrong reason?), **Brenk alerts** (does it carry a group chemists distrust?). An alert is a reason to look, not proof a compound is bad.

In [ ]:
# --- PROJECT 4: clean the library ------------------------------------------------------------
def rule_of_five_ok(m):
    """Lipinski's rule of five, with at most ONE of the four limits broken."""
    ok = [Descriptors.ExactMolWt(m) <= 500, Lipinski.NumHAcceptors(m) <= 10, Lipinski.NumHDonors(m) <= 5, Descriptors.MolLogP(m) <= 5]
    return sum(ok) >= 3

def catalog(name):
    p = FilterCatalogParams(); p.AddCatalog(getattr(FilterCatalogParams.FilterCatalogs, name)); return FilterCatalog(p)
PAINS, BRENK = catalog('PAINS'), catalog('BRENK')      # PAINS: assay interference. BRENK: 105 unwanted substructures.

funnel = {'molecules': len(df)}
lib = df[[rule_of_five_ok(m) for m in df.mol]];                        funnel['rule of five'] = len(lib)
lib = lib[[not PAINS.HasMatch(m) for m in lib.mol]];                   funnel['no PAINS'] = len(lib)

alerts = {}
flagged = []
for m in lib.mol:
    hits = [e.GetDescription() for e in BRENK.GetMatches(m)]
    flagged.append(bool(hits))
    for h in hits:
        alerts[h] = alerts.get(h, 0) + 1
lib = lib.assign(brenk_alert=flagged).reset_index(drop=True)
funnel['no Brenk alert'] = int((~lib.brenk_alert).sum())

for k, v in funnel.items():
    print(f'  {k:<18}{v:>8,}')
top_alerts = sorted(alerts.items(), key=lambda kv: -kv[1])[:6]
print('commonest alerts:', top_alerts)
results['p4'] = dict(funnel=funnel, top_alerts=top_alerts)

fig, ax = plt.subplots(figsize=(7, 3))
ax.barh(list(funnel)[::-1], list(funnel.values())[::-1], color='black')
for y, v in enumerate(list(funnel.values())[::-1]):
    ax.text(v, y, f' {v:,}', va='center')
ax.set_title(f'{TARGET_NAME}: molecules left after each filter'); plt.show()

# Check by hand: paste ten of these structures into swissadme.ch and see whether its alerts agree.
print(*lib[lib.brenk_alert].smiles.head(10), sep='\n')
# From here on, "the library" is the molecules that passed the rule of five and PAINS (the Brenk alerts stay as a flag, not a deletion).

## Project 5. See the chemistry
Fingerprints turn each molecule into a bit pattern. **Butina clustering** groups molecules whose fingerprints are close (distance 0.2 or less). The **maximum common substructure** is the biggest piece every molecule in a group shares: the scaffold, in words you can say out loud.

In [ ]:
# --- PROJECT 5: see the chemistry (clusters and shared cores) --------------------------------------
gen = rdFingerprintGenerator.GetRDKitFPGenerator(maxPath=5, fpSize=2048)
fps = [gen.GetFingerprint(m) for m in lib.mol]
dists = []
for i in range(1, len(fps)):                                     # distance = 1 - Tanimoto similarity
    dists.extend([1 - s for s in DataStructs.BulkTanimotoSimilarity(fps[i], fps[:i])])
clusters = sorted(Butina.ClusterData(dists, len(fps), 0.2, isDistData=True), key=len, reverse=True)   # molecules within distance 0.2 share a cluster
sizes = [len(c) for c in clusters]
print(f'{len(fps):,} molecules -> {len(clusters):,} clusters; {sum(s == 1 for s in sizes):,} are single molecules; the largest has {sizes[0]}')

def shared_core(mols, seconds=20):
    r = rdFMCS.FindMCS(mols, timeout=seconds, ringMatchesRingOnly=True)
    return r.numAtoms, r.numBonds, r.smartsString

big = [lib.mol[i] for i in clusters[0]]
atoms, bonds, core_smarts = shared_core(big)
potent = [m for m, p in zip(lib.mol, lib.pIC50) if p > 9]
pot_core = shared_core(potent) if len(potent) >= 2 else None
print(f'shared core of the largest cluster: {atoms} atoms, {bonds} bonds')
if pot_core: print(f'shared core of the {len(potent)} most potent molecules (pIC50 above 9): {pot_core[0]} atoms, {pot_core[1]} bonds')
results['p5'] = dict(molecules=len(fps), clusters=len(clusters), singletons=int(sum(s == 1 for s in sizes)), largest=sizes[0],
                     core_atoms_largest=atoms, potent_over_9=len(potent), core_atoms_potent=(pot_core[0] if pot_core else None))

# 20 diverse compounds to test first: the centre molecule of each of the 20 biggest clusters
picks = [lib.iloc[c[0]] for c in clusters[:20]]
Draw.MolsToGridImage([p.mol for p in picks], molsPerRow=5, subImgSize=(260, 200),
                     legends=[f'{p.molecule_chembl_id}  pIC50 {p.pIC50:.1f}' for p in picks])

## Project 6. Find analogues of a known drug
Donepezil is an approved acetylcholinesterase inhibitor. The first search shows why you clean before you search; the PubChem search shows that "75 percent similar" depends on the fingerprint behind the number.

In [ ]:
# --- PROJECT 6: find analogues of a known drug ------------------------------------------------------
DONEPEZIL = 'COC1=C(C=C2C(=C1)CC(C2=O)CC3CCN(CC3)CC4=CC=CC=C4)OC'      # PubChem CID 3152
query = Chem.MolFromSmiles(DONEPEZIL)
morgan = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
fq = morgan.GetFingerprint(query)

# Step 1: the raw search (salts and all). Look at the top of the list.
raw_sims = DataStructs.BulkTanimotoSimilarity(fq, [morgan.GetFingerprint(m) for m in df.mol])
top = df.assign(similarity=raw_sims).sort_values('similarity', ascending=False).head(12)
print('RAW search: the top hits are mostly the same drug in different forms')
display(top[['molecule_chembl_id', 'similarity', 'pIC50']].round(3))

# Step 2: strip salts (keep the biggest piece) and merge forms by InChIKey, then search again.
chooser = rdMolStandardize.LargestFragmentChooser()
df['clean'] = [chooser.choose(m) for m in df.mol]
df['inchikey'] = [Chem.MolToInchiKey(m) for m in df.clean]
merged = df.drop_duplicates('inchikey').copy()
merged['similarity'] = DataStructs.BulkTanimotoSimilarity(fq, [morgan.GetFingerprint(m) for m in merged.clean])
analogues = merged[merged.similarity < 0.999].sort_values('similarity', ascending=False).head(10)
print('CLEANED search: ten real analogues')
display(analogues[['molecule_chembl_id', 'similarity', 'pIC50']].round(3))
results['p6'] = dict(raw_top_ids=[[r.molecule_chembl_id, round(r.similarity, 3)] for r in top.head(4).itertuples()],
                     raw_hits_at_or_above_0_95=int((top.similarity >= 0.95).sum()), analogues_ten=[[r.molecule_chembl_id, round(r.similarity, 3), round(r.pIC50, 2)] for r in analogues.itertuples()])

# Step 3: PubChem's own 2D similarity search (75 percent threshold), then score every hit with the SAME fingerprint.
url = ('https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/fastsimilarity_2d/smiles/' + urllib.parse.quote(DONEPEZIL) +
       '/cids/JSON?Threshold=75&MaxRecords=10')
try:
    cids = get_json(url)['IdentifierList']['CID']
    props = get_json('https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/' + ','.join(map(str, cids)) + '/property/IsomericSMILES/JSON')['PropertyTable']['Properties']
    rows = []
    for p in props:
        m = Chem.MolFromSmiles(p.get('SMILES') or p.get('IsomericSMILES') or '')
        if m is not None:
            rows.append([p['CID'], round(DataStructs.TanimotoSimilarity(fq, morgan.GetFingerprint(m)), 3)])
    pub = pd.DataFrame(rows, columns=['PubChem CID', 'Morgan similarity'])
    print('PubChem says "75 percent similar". RDKit Morgan fingerprints say:')
    display(pub)
    results['p6']['pubchem_hits'] = len(pub); results['p6']['pubchem_low_under_0_35'] = int((pub['Morgan similarity'] < 0.35).sum())
except Exception as e:
    print('PubChem did not answer, try again later:', e)

---
# Your turn
**Task:** run the whole notebook again for CDK5 and compare the two funnels.

**Expected output:** a smaller dataset (a few hundred molecules), most of it surviving the filters, far fewer clusters.

In [ ]:
# --- YOUR TURN ---------------------------------------------------------------------------------------
# Change TARGET_NAME and TARGET_CHEMBL_ID in the settings cell to CDK5 / CHEMBL4036, then use Runtime > Restart and run all.
# Expected: far fewer molecules, a different funnel, a different largest cluster. Write the two funnels side by side.
print(json.dumps(results, indent=1, default=str)[:3000])

---
# Discussion
**What you learned:** where the numbers come from and how many molecules each cleaning step removes.

**What this cannot do, and where it breaks:** the 2 to 12 range is a judgement: a potency below 1 picomolar or above 10 millimolar is far more likely a unit or entry error than a result, but I cannot prove it for each one. ChEMBL mixes labs, assays and years, so potencies are not strictly comparable; the median of several records hides disagreement between them. Filters flag molecules, they do not judge them (roughly 5 percent of approved drugs contain a PAINS pattern). Similarity depends on the fingerprint. Counts drift because ChEMBL grows: **date every number you post**.